#### BUILD AGENT AI USING LANGCHAIN INSTEAD OF OOP AI

In [4]:
import os
import logging
from typing import List
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool, ToolRuntime
load_dotenv()

llm_api_url = os.getenv("LLM_API_URL")
llm_api_key = os.getenv("LLM_API_KEY")
llm_api_version = os.getenv("LLM_API_VERSION")
llm_model = os.getenv("LLM_MODEL")

logger = logging.getLogger(__name__)

In [ ]:
from json import tool



class AIChatBot:
    def __init__(self):
        self.system_prompt = "You are specialist Doctor in Cancer Disease"
        self.checkpointer = InMemorySaver()
        if llm_api_key:
            try:
                self.llm_model = init_chat_model(
                    model = llm_model
                    api_key=llm_api_key,
                    temperature=0.7,
                    base_url= llm_api_url)
            except Exception as e:
                raise Exception(f"Error in init_chat_model: {str(e)}")
            
        if self.llm_model:
            raise Exception(f"We have not done init_chat_model")
        model_with_tools = self.ai_chat_bot.
        self.ai_chat_bot = create_agent(
                model=self.llm_model,
                system_prompt=self.system_prompt
                checkpointer=self.checkpointer
        )
    def _build_message_chain(self, system_prompt_content: str, past_messages: List[Message], new_content: str) -> list:
        """
        Constructs the message chain for the LLM, including the system prompt,
        chat history, and the latest user query.
        """
        messages: List[BaseMessage] = [SystemMessage(content=system_prompt_content)]
        
        # Append historical messages
        for msg in past_messages:
            role = str(msg.role)
            content = str(msg.content)
            if role == "user":
                messages.append(HumanMessage(content=content))
            else:
                messages.append(AIMessage(content=content))
                
        # Append the latest user query
        messages.append(HumanMessage(content=new_content))
        
        return messages
    @tool
    def search_RAG(has_attachments, new_content, conv_id):
         if has_attachments:
            try:
                rag_context = search_knowledge_base(new_content, conv_id)
            except Exception as e:
                logger.error(f"Failed to retrieve RAG context: {e}")

        # Dynamically adjust the system prompt if RAG context is found
    def get_response(self, past_messages: List[Message], new_content: str, conv_id: int, has_attachments: bool) -> str:
        """
        Generates an AI response, utilizing RAG context if attachments exist in the conversation.
        """
        rag_context = ""
        system_prompt = self.default_system_prompt

        
        if rag_context:
            system_prompt = f"""You are a professional AI Docter assistant.
            Based on the EXTRACTED DOCUMENTS below, answer the user's question.
        - Always cite the source using [Page X] if you extract information from the documents.
        - If the information is not present in the documents, state honestly: "The provided documents do not mention this information." Do not fabricate data.

        --- EXTRACTED DOCUMENTS ---
        {rag_context}
        ---------------------------
        """
      
        messages = self._build_message_chain(system_prompt, past_messages, new_content)

        llm_model.bind_tools([self.search_RAG])

        try:
            logger.info("Invoking Azure OpenAI model...")
            response = self.ai_chat_bot.invoke(messages)
            return str(response.content)
        except Exception as e:
            logger.error(f"Azure OpenAI invocation failed: {str(e)}")
            raise Exception(f"AI Model connection error: {str(e)}")
    

In [5]:
from langchain.tools import tool
from langchain.chat_models import init_chat_model


model = init_chat_model(
    "claude-sonnet-4-6",
    temperature=0
)


# Define tools
@tool
def multiply(a: int, b: int) -> int:
    """Multiply `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a * b


@tool
def add(a: int, b: int) -> int:
    """Adds `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a + b


@tool
def divide(a: int, b: int) -> float:
    """Divide `a` and `b`.

    Args:
        a: First int
        b: Second int
    """
    return a / b


# Augment the LLM with tools
tools = [add, multiply, divide]
tools_by_name = {tool.name: tool for tool in tools}
model_with_tools = model.bind_tools(tools)

In [ ]:
from langchain.messages import AnyMessage
from typing_extensions import TypedDict, Annotated
import operator


class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]
    llm_calls: int

In [1]:
from pymilvus import MilvusClient, DataType, Function, FunctionType

client = MilvusClient(
    uri="http://localhost:19530",
)

schema = client.create_schema()

schema.add_field("id", DataType.INT64, is_primary=True, auto_id=False)

schema.add_field("document", DataType.VARCHAR, max_length=9000)

schema.add_field("dense", DataType.FLOAT_VECTOR, dim=1536)


{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'document', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 9000}}, {'name': 'dense', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1536}}], 'enable_dynamic_field': False, 'enable_namespace': False}

In [2]:
text_embedding_function = Function(
    name="azopenai",                                # Unique identifier for this embedding function
    function_type=FunctionType.TEXTEMBEDDING,       # Indicates a text embedding function
    input_field_names=["document"],                 # Scalar field(s) containing text data to embed
    output_field_names=["dense"],                   # Vector field(s) for storing embeddings
    params={                                        # Provider-specific embedding parameters
        "provider": "azure_openai",                 # Embedding provider name (must be "azure_openai")
        "model_name": "text-embedding-3-small",      # Model should be set to the deployment name you chose when you deployed the embedding model
        # Optional parameters (only specify if necessary):
        # "url": "https://{resource_name}.openai.azure.com/" # Optional: Your Azure OpenAI service endpoint
        # "credential": "apikey_dev",               # Optional: Credential label specified in milvus.yaml
        # "dim": "1536",                            # Optional: Shorten the output vector dimension
        # "user": "user123",                        # Optional: identifier for API tracking
    }
)

schema.add_function(text_embedding_function)


{'auto_id': False, 'description': '', 'fields': [{'name': 'id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'document', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 9000}}, {'name': 'dense', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1536}, 'is_function_output': True}], 'enable_dynamic_field': False, 'enable_namespace': False, 'functions': [{'name': 'azopenai', 'description': '', 'type': <FunctionType.TEXTEMBEDDING: 2>, 'input_field_names': ['document'], 'output_field_names': ['dense'], 'params': {'provider': 'azure_openai', 'model_name': 'text-embedding-3-small'}}]}

In [3]:
index_params = client.prepare_index_params()

index_params.add_index(
    field_name="dense",
    index_type="AUTOINDEX",
    metric_type="COSINE" 
)


In [4]:
client.create_collection(
    collection_name='demo', 
    schema=schema, 
    index_params=index_params
)


In [5]:
client.insert('demo', [
    {'id': 1, 'document': 'Milvus simplifies semantic search through embeddings.'},
    {'id': 2, 'document': 'Vector embeddings convert text into searchable numeric data.'},
    {'id': 3, 'document': 'Semantic search helps users find relevant information quickly.'},
])


{'insert_count': 3, 'ids': [1, 2, 3], 'cost': 0}

In [ ]:
load_dotenv()  # Load environment variables from .env file
open_api_key = os.getenv("GOOGLE_API_KEY")
model = init_chat_model("google_genai:gemini-2.5-flash-lite")
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

In [ ]:
response = model.invoke("Explain AI")
print(type(response))  # <class 'langchain.messages.AIMessage'>

In [ ]:
def get_weather(location: str) -> str:
    """Get the weather at a location."""
    ...

model_with_tools = model.bind_tools([get_weather])
response = model_with_tools.invoke("What's the weather in Paris?")

for tool_call in response.tool_calls:
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")
    print(f"ID: {tool_call['id']}")

In [ ]:
chunks = []
full_message = None
for chunk in model.stream("Hi"):
    chunks.append(chunk)
    print(chunk.text)
    full_message = chunk if full_message is None else full_message + chunk

In [ ]:
from langchain.messages import HumanMessage

# String content
human_message = HumanMessage("Hello, how are you?")

# Provider-native format (e.g., OpenAI)
human_message = HumanMessage(content=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image_url", "image_url": {"url": "https://example.com/image.jpg"}}
])

# List of standard content blocks
human_message = HumanMessage(content_blocks=[
    {"type": "text", "text": "Hello, how are you?"},
    {"type": "image", "url": "https://example.com/image.jpg"},
])

In [ ]:
from langchain.messages import AIMessage
from langchain.messages import ToolMessage

# After a model makes a tool call
# (Here, we demonstrate manually creating the messages for brevity)
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)

# Execute tool and create result message
weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  # Must match the call ID
)

# Continue conversation
messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  # Model's tool call
    tool_message,  # Tool execution result
]
response = model.invoke(messages)  # Model processes the result

In [ ]:
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="google_genai:gemini-2.5-flash-lite",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in New York?"}]}
)


In [ ]:

SYSTEM_PROMPT = """You are an expert weather forecaster, who speaks in puns.

You have access to two tools:

- get_weather_for_location: use this to get the weather for a specific location
- get_user_location: use this to get the user's location

If a user asks you for the weather, make sure you know the location. If you can tell from the question that they mean wherever they are, use the get_user_location tool to find their location."""

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@tool
def get_weather_for_location(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

@dataclass
class Context:
    """Custom runtime context schema."""
    user_id: str

@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """Retrieve user information based on user ID."""
    user_id = runtime.context.user_id
    return "Florida" if user_id == "1" else "SF"

# Configure your model  


In [ ]:
from langchain.chat_models import init_chat_model

load_dotenv()  # Load environment variables from .env file
claude_api_key = os.getenv("CLAUDE_API_KEY")
model = init_chat_model(
    "google_genai:gemini-2.5-flash-lite",
    api_key=open_api_key,
    temperature=0.9,
    max_tokens=1000   
)

### Define response format  

In [ ]:
from dataclasses import dataclass

# We use a dataclass here, but Pydantic models are also supported.
# @dataclass
# class ResponseFormat:
#     """Response schema for the agent."""
#     # A punny response (always required)
#     punny_response: str
#     # Any interesting information about the weather if available
#     weather_conditions: str | None = None

Add memory to help Ai to remmeber previous conservations and context


In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
checkpointer = InMemorySaver()

In [ ]:
from langchain.agents.structured_output import ToolStrategy

agent = create_agent(
    model=model,
    system_prompt= SYSTEM_PROMPT,
    tools=[get_weather_for_location, get_user_location],
    context_schema=Context,
    # response_format=ToolStrategy(ResponseFormat),
    checkpointer=checkpointer
    
)

config = {"configurable": {"thread_id": "user_1"}}

# response = agent.invoke(
#     {"messages": [{"role": "user", "content": "what is the weather outside in Japan?"}]},
#     config=config,
#     context=Context(user_id="1")    
# )
for chunk in agent.stream({"messages": [{"role": "user", "content": "what is the weather outside in Japan?"}]},
                          config=config, context=Context(user_id="1")):
    print(chunk, end="|", flush=True)


# print(response['structured_response'])

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def state_based_tools(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse]
) -> ModelResponse:
    """Filter tools based on conversation State."""
    # Read from State: check if user has authenticated
    state = request.state
    is_authenticated = state.get("authenticated", False)
    message_count = len(state["messages"])

    # Only enable sensitive tools after authentication
    if not is_authenticated:
        tools = [t for t in request.tools if t.name.startswith("public_")]
        request = request.override(tools=tools)
    elif message_count < 5:
        # Limit tools early in conversation
        tools = [t for t in request.tools if t.name != "advanced_search"]
        request = request.override(tools=tools)

    return handler(request)

agent = create_agent(
    model="gpt-4.1",
    tools=[public_search, private_search, advanced_search],
    middleware=[state_based_tools]
)

In [ ]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

literary_agent = create_agent(
    model="anthropic:claude-sonnet-4-5",
    system_prompt=SystemMessage(
        content=[
            {
                "type": "text",
                "text": "You are an AI assistant tasked with analyzing literary works.",
            },
            {
                "type": "text",
                "text": "<the entire contents of 'Pride and Prejudice'>",
                "cache_control": {"type": "ephemeral"}
            }
        ]
    )
)

result = literary_agent.invoke(
    {"messages": [HumanMessage("Analyze the major themes in 'Pride and Prejudice'.")]}
)

Dynamic System Prompt

In [ ]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent = create_agent(
    model="gpt-4.1",
    tools=[web_search],
    middleware=[user_role_prompt],
    context_schema=Context
)

# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "expert"}
)

# Cách tạo name cho node fro multi-agent 

In [ ]:
agent = create_agent(
    model,
    tools,
    name="research_assistant"
)

Phương pháp streaming kết quả 

In [ ]:
from langchain.messages import AIMessage, HumanMessage

for chunk in agent.stream({
    "messages": [{"role": "user", "content": "Search for AI news and summarize the findings"}]
}, stream_mode="values"):
    # Each chunk contains the full state at that point
    latest_message = chunk["messages"][-1]
    if latest_message.content:
        if isinstance(latest_message, HumanMessage):
            print(f"User: {latest_message.content}")
        elif isinstance(latest_message, AIMessage):
            print(f"Agent: {latest_message.content}")
    elif latest_message.tool_calls:
        print(f"Calling tools: {[tc['name'] for tc in latest_message.tool_calls]}")

In [ ]:
response = model.invoke("What is the capital of France?", logprobs=True)

In [ ]:
ai_model = init_chat_model(
    model ="google_genai:gemini-2.5-flash-lite",
    api_key=open_api_key,
).bind(logprob=True)

response = ai_model.invoke("Why do parrots talk?")
print(response.response_metadata["logprobs"])

backend/milvus_db.py

In [ ]:
from pymilvus import connections, FieldSchema, CollectionSchema, DataType, Collection, utility

# Cấu hình kết nối
MILVUS_HOST = "localhost"
MILVUS_PORT = "19530" # Cổng mặc định của Milvus Docker
COLLECTION_NAME = "chatbot_knowledge_base"
VECTOR_DIM = 384 # Kích thước vector của model 'all-MiniLM-L6-v2' (mô hình miễn phí siêu nhanh)

def init_milvus():
    """Hàm khởi tạo kết nối và tạo Collection nếu chưa có"""
    print("Đang kết nối đến Milvus...")
    connections.connect("default", host=MILVUS_HOST, port=MILVUS_PORT)
    
    # Kiểm tra xem Collection đã tồn tại chưa
    if utility.has_collection(COLLECTION_NAME):
        print(f"✅ Collection '{COLLECTION_NAME}' đã tồn tại. Bỏ qua bước tạo mới.")
        return Collection(COLLECTION_NAME)

    print(f"🚀 Đang tạo mới Collection '{COLLECTION_NAME}'...")
    
    # 1. Định nghĩa các cột (Fields) cho Collection
    fields = [
        # Cột ID tự động tăng
        FieldSchema(name="id", dtype=DataType.INT64, is_primary=True, auto_id=True),
        
        # Cột chứa đoạn text (đã cắt nhỏ từ PDF)
        FieldSchema(name="text", dtype=DataType.VARCHAR, max_length=65535),
        
        # Cột METADATA 1: Dùng để lọc theo phòng chat
        FieldSchema(name="conversation_id", dtype=DataType.INT64),
        
        # Cột METADATA 2: Dùng để quản lý file (sau này muốn xóa 1 file cho dễ)
        FieldSchema(name="file_id", dtype=DataType.INT64),
        
        # Cột quan trọng nhất: Chứa dãy số Vector của đoạn text
        FieldSchema(name="vector", dtype=DataType.FLOAT_VECTOR, dim=VECTOR_DIM)
    ]
    
    schema = CollectionSchema(fields, description="Kho tri thức RAG cho Chatbot")
    
    # 2. Tạo Collection
    collection = Collection(name=COLLECTION_NAME, schema=schema)
    
    # 3. Đánh chỉ mục (Index) để tìm kiếm cực nhanh
    index_params = {
        "metric_type": "L2", # Đo khoảng cách vector bằng L2 (Euclidean)
        "index_type": "IVF_FLAT", # Kiểu index phổ biến
        "params": {"nlist": 128}
    }
    collection.create_index(field_name="vector", index_params=index_params)
    
    # 4. Load collection vào RAM để sẵn sàng query
    collection.load()
    print(f"✅ Đã tạo và load thành công Collection '{COLLECTION_NAME}'!")
    
    return collection

# Chạy thử nếu gọi trực tiếp file này
if __name__ == "__main__":
    init_milvus()

services/rag_service.py

In [ ]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from pymilvus import Collection, connections
from ..milvus_db import MILVUS_HOST, MILVUS_PORT, COLLECTION_NAME

# 1. Khởi tạo model nhúng (Embedding) cục bộ. 
# Model này siêu nhẹ, siêu nhanh, và đặc biệt là chạy offline 100% bảo mật.
embedder = SentenceTransformer('all-MiniLM-L6-v2')

def process_and_store_pdf(file_path: str, conversation_id: int, attachment_id: int):
    """
    Hàm này đọc file PDF, cắt nhỏ chữ, biến thành Vector và nhét vào Milvus
    """
    print(f"🔄 Bắt đầu xử lý file: {file_path}")
    
    # 2. Đọc chữ từ file PDF
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    
    if not documents:
        print("❌ Lỗi: File PDF trống hoặc không thể đọc chữ.")
        return False

    # 3. Cắt nhỏ văn bản (Chunking)
    # Cắt mỗi đoạn khoảng 500 ký tự, cho gối lên nhau 50 ký tự để không bị đứt câu
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, 
        chunk_overlap=50, 
        separators=["\n\n", "\n", ".", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    
    # Rút trích phần text từ các chunks
    texts = [chunk.page_content for chunk in chunks]
    print(f"✂️ Đã cắt file thành {len(texts)} đoạn nhỏ.")

    # 4. Biến chữ thành số (Embedding)
    print("🧠 Đang mã hóa văn bản thành Vector...")
    # Quá trình này dùng CPU của bạn để tính toán ma trận
    vectors = embedder.encode(texts).tolist() 

    # 5. Chuẩn bị dữ liệu theo đúng chuẩn Schema đã định nghĩa ở milvus_db.py
    conversation_ids = [conversation_id] * len(texts)
    file_ids = [attachment_id] * len(texts)
    
    # Danh sách các cột cần chèn (Bỏ qua cột 'id' vì auto_id=True)
    data_to_insert = [
        texts,             # Cột 1: text
        conversation_ids,  # Cột 2: conversation_id (Dùng để lọc lúc user hỏi)
        file_ids,          # Cột 3: file_id
        vectors            # Cột 4: vector (Cái này là linh hồn của Milvus)
    ]

    # 6. Mở cổng kết nối Milvus và "Tống" dữ liệu vào
    connections.connect("default", host=MILVUS_HOST, port=MILVUS_PORT)
    collection = Collection(COLLECTION_NAME)
    
    print("💾 Đang lưu vào database Milvus...")
    insert_result = collection.insert(data_to_insert)
    
    # Bắt buộc phải có dòng này để Milvus sắp xếp lại index, cho phép tìm kiếm ngay lập tức
    collection.flush()
    
    print(f"✅ HOÀN TẤT! Đã lưu thành công {insert_result.insert_count} vectors vào Milvus.")
    return True

routers/attachment.py

In [ ]:
# 1. Bổ sung import BackgroundTasks
from fastapi import APIRouter, Depends, HTTPException, status, UploadFile, File, BackgroundTasks
# ... các import khác giữ nguyên
from ..services.rag_service import process_and_store_pdf # <--- Import hàm xử lý PDF

# ... (Khúc khai báo router giữ nguyên) ...

@router.post("/", response_model=schemas.AttachmentOut, status_code=status.HTTP_201_CREATED)
def upload_file(
    conv_id: int, 
    file: UploadFile = File(...), 
    db: Session = Depends(get_db), 
    current_user: models.User = Depends(oauth2.get_current_user),
    background_tasks: BackgroundTasks = BackgroundTasks() # <--- Thêm tham số này
):
    # ... (Khúc kiểm tra quyền và lưu file vào ổ cứng giữ nguyên y hệt lúc nãy) ...
    # (Đoạn shutil.copyfileobj...)

    # Lưu vào Database SQL
    new_attachment = models.Attachment(
        conversation_id=conv_id,
        file_name=file.filename,
        file_path=file_path
    )
    
    db.add(new_attachment)
    db.commit()
    db.refresh(new_attachment)

    # ĐÂY LÀ PHÉP THUẬT: Đẩy việc xử lý Milvus chạy ngầm ở Background
    background_tasks.add_task(
        process_and_store_pdf, 
        file_path=file_path, 
        conversation_id=conv_id, 
        attachment_id=new_attachment.id
    )

    return new_attachment

services/rag_service.py

In [ ]:
# (Giữ nguyên các import và hàm process_and_store_pdf ở trên) ...

def search_knowledge_base(query: str, conversation_id: int, top_k: int = 3) -> str:
    """
    Tìm kiếm câu trả lời trong Milvus dựa trên câu hỏi và ID phòng chat.
    top_k: Số lượng đoạn văn bản liên quan nhất muốn lấy ra (mặc định lấy 3 đoạn).
    """
    try:
        print(f"🔍 Đang lục lọi Milvus cho câu hỏi: '{query}'")

        # 1. Biến câu hỏi của User thành Vector để so khớp
        query_vector = embedder.encode([query]).tolist()

        # 2. Kết nối Milvus
        connections.connect("default", host=MILVUS_HOST, port=MILVUS_PORT)
        collection = Collection(COLLECTION_NAME)
        collection.load()

        # 3. TÌM KIẾM (Đỉnh cao của Metadata Filtering nằm ở đây)
        search_params = {
            "metric_type": "L2",
            "params": {"nprobe": 10}
        }
        
        # 'expr' chính là bộ lọc: Chỉ lấy vector thuộc về conversation_id này!
        results = collection.search(
            data=query_vector,
            anns_field="vector",
            param=search_params,
            limit=top_k,
            expr=f"conversation_id == {conversation_id}", 
            output_fields=["text"] # Nhớ dặn Milvus trả về nội dung chữ
        )

        # 4. Gom các đoạn text tìm được lại thành 1 cục văn bản
        context = ""
        if results and len(results[0]) > 0:
            print(f"🎯 Tìm thấy {len(results[0])} đoạn thông tin mật!")
            for hit in results[0]:
                context += hit.entity.get("text") + "\n\n"
        
        return context

    except Exception as e:
        print(f"⚠️ Milvus rỗng hoặc có lỗi: {e}")
        return "" # Nếu lỗi thì trả về chuỗi rỗng để AI chat bình thường

services/ai_bot.py

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.schema import HumanMessage, SystemMessage, AIMessage
from ..config import settings
from ..models import Message
from typing import List

# IMPORT hàm search từ file bên cạnh
from .rag_service import search_knowledge_base 

class AIChatBot:
    def __init__(self):
        self.llm = ChatGoogleGenerativeAI(
            model="gemini-3.1-flash", # Thay tên model nếu bạn đang xài bản khác
            google_api_key=settings.google_api_key
        )
        self.normal_prompt = "Bạn là một trợ lý AI hữu ích, nói tiếng Việt chuẩn và ngắn gọn."

    def get_response(self, past_messages: List[Message], new_content: str, conv_id: int, has_attachments: bool) -> str:
        """
        Nhận vào thêm conv_id và biến has_attachments (True/False) từ Database.
        """
        messages = []
        rag_context = ""

        # LÔ-GIC CHUẨN CỦA BẠN: Nếu có file PDF thì mới đi mò Milvus
        if has_attachments:
            rag_context = search_knowledge_base(new_content, conv_id)

        # Định hình lại TÍNH CÁCH (System Prompt) của AI
        if rag_context != "":
            # NẾU TÌM THẤY DỮ LIỆU RAG -> Ép AI thành chuyên gia đọc tài liệu
            rag_prompt = f"""Bạn là một chuyên gia phân tích tài liệu xuất sắc.
Dựa vào các THÔNG TIN TỪ TÀI LIỆU DƯỚI ĐÂY, hãy trả lời câu hỏi của người dùng. 
Nếu thông tin không có trong tài liệu, hãy nói "Tài liệu đính kèm không đề cập đến vấn đề này" và có thể bổ sung thêm kiến thức của bạn.

--- THÔNG TIN TỪ TÀI LIỆU ---
{rag_context}
-----------------------------
"""
            messages.append(SystemMessage(content=rag_prompt))
        else:
            # NẾU KHÔNG CÓ FILE (hoặc file không liên quan) -> Trò chuyện bình thường
            messages.append(SystemMessage(content=self.normal_prompt))
            
        # Nạp lịch sử cũ
        for msg in past_messages:
            if msg.role == "user":
                messages.append(HumanMessage(content=msg.content))
            else:
                messages.append(AIMessage(content=msg.content))
                
        # Nạp câu hỏi mới
        messages.append(HumanMessage(content=new_content))

        # Gọi "Bộ não"
        try:
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            raise Exception(f"Lỗi AI: {str(e)}")

routers/message.py

In [ ]:
# ... (code lấy lịch sử cũ giữ nguyên) ...

    # [CODE THÊM MỚI] - Kiểm tra xem phòng chat này có file PDF nào không?
    attachment_count = db.query(models.Attachment).filter(models.Attachment.conversation_id == conv_id).count()
    has_attachments = attachment_count > 0 

    # 3. NHỜ SERVICE XỬ LÝ AI
    try:
        # Nhớ truyền thêm conv_id và has_attachments vào nhé!
        ai_response_content = ai_service.get_response(
            past_messages=past_messages, 
            new_content=message_data.content, 
            conv_id=conv_id, 
            has_attachments=has_attachments
        )
    except Exception as e:
        raise HTTPException(status_code=status.HTTP_500_INTERNAL_SERVER_ERROR, detail=str(e))

    # ... (code lưu database giữ nguyên) ...

In [ ]:
from langchain_openai import OpenAIEmbeddings
OpenAIEmbeddings(
                    base_url=settings.EMBED_API_URL,
                    api_key=settings.EMBED_API_KEY,
                    model=settings.EMBED_MODEL,
                )